In [31]:
import sys
from pathlib import Path
# Add project root to path
project_root = Path('/work')
sys.path.append(str(project_root))

import re

def load_corpus_documents(file_base: str) -> tuple:
    """
    Load both texto and resumen documents.
    
    Args:
        file_base: Base filename for documents
        
    Returns:
        Tuple of (texto_docs, resumen_docs)
    """
    print("Loading document collections...")
    
    # Load texto documents
    texto_path = Path(f'/work/data/processed/{file_base}_texto.pkl')
    texto_docs = load_documents(texto_path)
    
    if texto_docs is None:
        raise FileNotFoundError(f"Could not load texto documents from {texto_path}")
    
    # Load resumen documents
    resumen_path = Path(f'/work/data/processed/{file_base}_resumen.pkl')
    resumen_docs = load_documents(resumen_path)
    
    if resumen_docs is None:
        raise FileNotFoundError(f"Could not load resumen documents from {resumen_path}")
    
    print(f"Loaded {len(texto_docs)} texto documents")
    print(f"Loaded {len(resumen_docs)} resumen documents")
    
    return texto_docs, resumen_docs

def extract_compound_terms_from_documents(documents):
    """
    Extract all compound terms (multi-word terms) from a list of Document objects.
    
    Args:
        documents: List of Document objects with metadata containing 'parameters'
        
    Returns:
        List of unique compound terms, sorted alphabetically
    """
    compound_terms = set()  # Use set to avoid duplicates
    
    def is_compound_term(text):
        """Check if a term has more than one word"""
        if not text or not isinstance(text, str):
            return False
        
        # Clean the text: remove extra spaces and trim
        clean_text = re.sub(r'\s+', ' ', text.strip())
        
        # Check if it contains more than one word
        words = [word for word in clean_text.split(' ') if word]
        return len(words) > 1
    
    def add_if_compound(text):
        """Add text to compound_terms set if it's a compound term"""
        if is_compound_term(text):
            clean_text = re.sub(r'\s+', ' ', text.strip())
            compound_terms.add(clean_text)
    
    # Iterate through each Document object
    for document in documents:
        # Get parameters from document metadata
        parameters = document.metadata.get('parameters', {})
        
        # Skip if no parameters or parameters is empty
        if not parameters:
            continue
        
        # Iterate through each parameter (A, B, C, etc.)
        for param_key, parameter in parameters.items():
            
            # Check parameter label
            if 'label' in parameter:
                add_if_compound(parameter['label'])
            
            # Check values array
            if 'values' in parameter and isinstance(parameter['values'], list):
                for value_item in parameter['values']:
                    # Check value field
                    if 'value' in value_item:
                        add_if_compound(value_item['value'])
    
    # Convert set to sorted list
    return sorted(list(compound_terms))

def main(file_base):
    texto_docs, resumen_docs = load_corpus_documents(file_base)
    compound_terms = extract_compound_terms_from_documents(texto_docs)
    return compound_terms


In [32]:
compound_terms =main('OEB')

Loading document collections...
Loaded 47514 texto documents
Loaded 47514 resumen documents


In [33]:
compound_terms

['0,80 a 1,00 m',
 '1 o 2',
 '1,10 m',
 "3 1/2''",
 '3 <= i < 5 horas',
 'BANDA DE MANTENIMIENTO',
 'Bajo vías',
 'CONDICIONES DE EJECUCIÓN',
 'Con topo',
 'Cruce de carretera',
 'Cualquier condición de ejecución',
 'Cualquier franja horaria',
 'Cualquier franja horaria excepcional',
 'Diurno Excepcional',
 'Diurno excepcional',
 'No aplica',
 'No necesita intervalo',
 'Nocturno Excepcional',
 'Nocturno excepcional',
 'Nº TUBOS',
 'Suministro y montaje',
 'TIPO DE ACCIÓN',
 'TIPO DE TERRENO',
 'Volumen escaso',
 'Volumen relevante',
 'con reposición',
 'de cantera',
 'de la traza',
 'hasta 0,80 m',
 'i < 3 horas',
 'i >= 5 horas',
 'i< 3 horas',
 'sin reposición']